# Chapter 11: Production Capstone -- The Autonomous CI Triage Bot

> *"Theory without practice is sterile, but practice without theory is blind."*
> -- Immanuel Kant

Across the preceding ten chapters, we developed the dual mathematical
engines of **Bayesian Inference** and **Information Theory**:
- **Ch 00**: Surprise as asymmetric currency ($I(x) = -\log_2 P(x)$).
- **Ch 01**: Shannon entropy $H(p)$ and the Beta-Binomial conjugate update.
- **Ch 02**: Relative entropy $D_{\text{KL}}(P \parallel Q)$ and the wasted bits
  of misaligned priors.
- **Ch 03**: E.T. Jaynes' Principle of Maximum Entropy for fleet baseline
  construction.
- **Ch 04**: Information Venn geometry ($H(X, Y), I(X; Y), \text{VI}$) and
  causal DAGs.
- **Ch 05**: Curvature, Laplace approximations, Fisher information, and
  Information Geometry.
- **Ch 06**: Markov Chain Monte Carlo (MCMC), typical sets, and entropy
  production.
- **Ch 07**: Compression and partial pooling via the Information Bottleneck
  ($\min I(X; T) - \beta I(T; Y)$).
- **Ch 08**: Bayesian model comparison as code efficiency (WAIC, PSIS-LOO, and
  Pareto $k$).
- **Ch 09**: Bayesian decision theory, asymmetric loss functions, and Wald SPRT.
- **Ch 10**: Active learning via Expected Information Gain (EIG), BALD, and
  Horstein probabilistic bisection.

In this Capstone chapter, we assemble every single piece of theory into a
unified, production-grade engineering system: **The Autonomous CI Triage
Bot (`CITriageEngine`)**.

We demonstrate how a single production system tracks non-stationary flakiness
without inertia, gates quarantine entry and exit via posterior predictive
streaks, isolates infrastructure crashes via Pareto $k$, dynamically
terminates PR retries via Wald SPRT, and actively bisects broken commits
using Horstein search -- saving hundreds of thousands of dollars in cloud
infrastructure while protecting customer reliability.


## 1. System Architecture: The Full Lifecycle of a Flaky Test

In large-scale continuous integration, a test is not an isolated binary check;
it is a living stochastic process that moves through a well-defined lifecycle:

![Autonomous CI Triage Bot End-to-End Pipeline Architecture](../images/ch11_autonomous_triage_pipeline.png)


## 2. Stage 1: Non-Stationary Bayesian Tracking with Exponential Forgetting ($\gamma$)

### The Inertia Problem of Static Bayes
Standard conjugate Bayesian updating is cumulative:

$$
\alpha_t = \alpha_{t-1} + y_t, \qquad \beta_t = \beta_{t-1} + (1 - y_t)
$$

In static scientific experiments, accumulation is ideal. But continuous
software integration is **fundamentally non-stationary**:
- A mature integration test may pass $2{,}000$ consecutive times over three
  months without a single failure ($D_0 = \text{Beta}(1, 2001)$, mean flakiness
  $\approx 0.05\%$).
- On Wednesday morning, an upstream Linux kernel patch introduces a $20\%$
  timing race condition ($p = 0.20$).
- If we apply static Bayesian updating, how long does it take for the
  posterior mean to cross an SLA threshold of $5\%$ ($0.05$)?

$$
\mathbb{E}[\theta_B] = \frac{1 + 2B}{2002 + 10B} \ge 0.05 \implies 1 + 2B \ge 100.1 + 0.5B \implies 1.5B \ge 99.1 \implies B \ge 67 \text{ batches}
$$

Static Bayes requires **67 consecutive batches** ($670$ test runs and $134$
broken builds!) before its posterior mean even reaches $5\%$.
Because past evidence is weighted equally to today's evidence, static Bayes
suffers from crippling **inertia**, blinding the team to acute outages.

### The Discounted Beta-Binomial Filter
To model non-stationary flakiness, we introduce an **exponential forgetting
factor** $\gamma \in (0, 1)$:

$$
\alpha_t = \gamma \alpha_{t-1} + k_t, \qquad \beta_t = \gamma \beta_{t-1} + (m_t - k_t)
$$

where $m_t$ is the batch size and $k_t$ is the number of observed failures.
The decay factor $\gamma$ is calibrated to a physical half-life $t_{1/2}$ in
batches:

$$
\gamma = 0.5^{1 / t_{1/2}} \iff t_{1/2} = \frac{-\ln 2}{\ln \gamma}
$$

In steady state, if batches of $m$ runs arrive regularly, the total effective
pseudo-count sample memory is bounded by the geometric series:

$$
N_{\text{eff}} = \sum_{j=0}^\infty \gamma^j m = \frac{m}{1 - \gamma}
$$

For $m = 10$ runs per batch and $\gamma = 0.95$:
- $N_{\text{eff}} = \frac{10}{1 - 0.95} = 200$ test runs ($20$ batches).
- Half-life $t_{1/2} = \frac{-\ln 2}{\ln 0.95} \approx 13.5$ batches.

### Concrete Numerical Comparison: Static ($\gamma = 1.0$) vs Discounted ($\gamma = 0.95$)
Consider our mature test ($\text{mean} \approx 0.05\%$) experiencing a $20\%$
regression ($k_t = 2$ failures per batch of $m_t = 10$ runs).
Let baseline discounted memory be $\alpha_0 = 1.0, \beta_0 = 200.0$
($N_{\text{eff}} = 200$).

| Batch $t$ | Metric | Static Bayes ($\gamma = 1.0$) | Discounted Bayes ($\gamma = 0.95$) | Operational Assessment |
| :---: | :--- | :--- | :--- | :--- |
| **Batch 1** | Posterior $(\alpha_1, \beta_1)$ | $(3.0, 2009.0)$ | $(2.95, 198.00)$ | Static mean is $0.15\%$; Discounted is $1.47\%$. |
| | Tail Risk $P(\theta > 0.05)$ | $0.00\%$ ($1.2 \times 10^{-13}$) | $0.22\%$ | Discounted responds immediately; static is dormant. |
| | Baseline $D_{\text{KL}}$ Shift | $1.650$ bits | $1.620$ bits | Surprise shift signals anomaly onset. |
| **Batch 2** | Posterior $(\alpha_2, \beta_2)$ | $(5.0, 2017.0)$ | $(4.80, 196.10)$ | Static mean is $0.25\%$; Discounted climbs to $2.39\%$. |
| | Tail Risk $P(\theta > 0.05)$ | $0.00\%$ ($1.7 \times 10^{-10}$) | $2.19\%$ | Discounted tail risk multiplies by $10\times$. |
| | Baseline $D_{\text{KL}}$ Shift | $4.055$ bits | $3.911$ bits | Strong information-theoretic evidence of shift. |
| **Batch 3** | Posterior $(\alpha_3, \beta_3)$ | $(7.0, 2025.0)$ | $(6.56, 194.29)$ | Static mean is $0.34\%$; Discounted reaches $3.27\%$. |
| | Tail Risk $P(\theta > 0.05)$ | $0.00\%$ ($1.5 \times 10^{-8}$) | **$9.38\%$** | Discounted crosses early warning threshold. |
| | Baseline $D_{\text{KL}}$ Shift | $6.604$ bits | **$6.259$ bits** | **Acute Alert**: Massive parameter shift confirmed. |

By Batch 3 (only 30 test runs!), the relative entropy shift exceeds $6.2$
bits, and discounted tail risk has leaped to $9.38\%$. Within 7 batches,
discounted Bayes triggers quarantine, while static Bayes remains blind for 67
batches!


## 3. Stage 2: Cost-Optimal Quarantine Entry & Exit (Posterior Predictive Streak Gate)

### The Economic Cost of Test Disruption
Leaving a flaky test on the blocking critical path imposes a severe daily Bayes
Risk:

$$
\text{Risk}_{\text{blocking}}(\theta) = C_{\text{block}} \cdot \theta \cdot N_{\text{PR}}
$$

where $C_{\text{block}}$ is developer wait cost per broken PR and
$N_{\text{PR}}$
is daily PR volume. Conversely, quarantining the test moves execution to an
asynchronous shadow fleet, costing a fixed monitoring overhead
$C_{\text{quarantine}}$.

### Quarantine Entry Gate: Dual Tail-Risk and Epistemic Confidence
To prevent alert fatigue from transient noise flukes, quarantine entry requires
satisfying two simultaneous statistical constraints:
1. **SLA Tail Risk Gate**: The posterior probability that failure rate exceeds
   the SLA threshold $\tau_{\text{SLA}} = 0.05$ must be at least $90\%$:

$$
P(\theta_t > \tau_{\text{SLA}} \mid \mathcal{D}_t) = 1 - I_{\tau_{\text{SLA}}}(\alpha_t, \beta_t) \ge 0.90
$$

2. **Epistemic Certainty Gate**: The differential entropy $h(\theta)$ of the
   Beta posterior must correspond to an effective support width $W_{\text{eff}}$
   of at most $0.15$:

$$
W_{\text{eff}} = 2^{h(\theta_t)} \le 0.15
$$

where $h(\text{Beta}(\alpha, \beta)) = \frac{\ln B(\alpha, \beta) -
(\alpha-1)\psi(\alpha) - (\beta-1)\psi(\beta) +
(\alpha+\beta-2)\psi(\alpha+\beta)}{\ln 2}$ (in bits).
This ensures quarantine is triggered only when we are confident the test is
genuinely flaky, not when we have high uncertainty from a small sample.

### Quarantine Exit: The Beta-Binomial Posterior Predictive Streak Gate
When an engineer pushes a purported fix to the quarantined test, how do we
decide it is safe to graduate back to blocking PR merges?

In industry, teams often rely on naive heuristics like *"it passed 5 times in a
row."* But for a test with $10\%$ flakiness, the probability of passing 5 runs
in a row by pure chance is $(0.90)^5 = 59.0\%$! Naive heuristics graduate
unfixed flaky tests more than half the time.

The autonomous triage engine enforces the **Beta-Binomial Posterior Predictive
Streak Gate**:
Given current posterior parameters $(\alpha_t, \beta_t)$ following shadow soak
runs, what is the exact probability that the test will pass its next $k = 25$
consecutive runs in production without breaking a single PR?

Integrating over our posterior uncertainty in $\theta$:

$$
P(\text{pass next } k \text{ runs} \mid \alpha_t, \beta_t) = \int_0^1 (1 - \theta)^k \frac{\theta^{\alpha_t - 1} (1 - \theta)^{\beta_t - 1}}{B(\alpha_t, \beta_t)} d\theta = \frac{B(\alpha_t, \beta_t + k)}{B(\alpha_t, \beta_t)}
$$

Expanding via the gamma function product property:

$$
P(\text{pass next } k=25 \text{ runs} \mid \alpha_t, \beta_t) = \prod_{j=0}^{24} \frac{\beta_t + j}{\alpha_t + \beta_t + j} \ge 0.90
$$

### Bayesian A/B Fix Verification Gate
Simultaneously, the triage engine executes a two-sample Bayesian A/B hypothesis
test comparing the pre-fix flakiness $\theta_{\text{pre-fix}} \sim
\text{Beta}(\alpha_A, \beta_A)$
against the post-fix soak flakiness $\theta_{\text{post-fix}} \sim
\text{Beta}(\alpha_B, \beta_B)$:

$$
P(\theta_{\text{post-fix}} < \theta_{\text{pre-fix}} \mid \mathcal{D}) = \mathbb{E}_{\theta_A, \theta_B}[\mathbb{I}(\theta_B < \theta_A)] \ge 0.95
$$

Only when **both** the Streak Probability $\ge 90\%$ and the Fix Probability
$\ge 95\%$ are met does the engine auto-graduate the test back to blocking.


## 4. Mathematical Synthesis of Pipeline Stages

### Stage 3: Outlier Gating via PSIS-LOO Pareto $k$
When a build fails, before penalizing the test author, we evaluate the Pareto
shape parameter $k_i$ from PSIS-LOO (Ch 08). If $k_i \ge 0.7$, the observation
possesses infinite importance-sampling variance -- it represents an out-of-band
infrastructure crash (host OOM, machine reboot, network split). The engine
routes the ticket to SRE and removes the run from developer scorecards.

### Stage 4: Wald SPRT Dynamic Execution Engine
For active PR validation, the bot runs Wald's Sequential Probability Ratio
Test (Ch 09):

$$
S_m = S_{m-1} + \ln \frac{p(y_m \mid H_1)}{p(y_m \mid H_0)}
$$

with asymmetric error boundaries calibrated to business costs:

$$
A = \ln\left(\frac{1 - \beta_{\text{err}}}{\alpha_{\text{err}}}\right), \qquad B = \ln\left(\frac{\beta_{\text{err}}}{1 - \alpha_{\text{err}}}\right)
$$

Terminates retries dynamically in an average of $1.15$ runs, cutting cloud
compute by over $60\%$.

### Stage 5: Horstein Probabilistic Culprit Bisection
When a broken batch merges, the bot checks out commits along the probabilistic
median of belief $c_t^* = \arg\min_c |F_t(c) - 0.50|$ (Ch 10). Soft updates
ensure that even a $30\%$ flakiness channel isolates the broken commit in
$O(\frac{\log K}{1 - H_2(\epsilon)})$ runs without catastrophic wrong-branch
traps.


## 5. What Do We Actually Do With This Number? (Production Deployment Runbook)

### Metric 1: Fleet Flakiness Entropy Debt ($H_{\text{fleet}}$)
- **Calculation**: $H_{\text{fleet}} = \sum_{j=1}^J H_2(\tilde{\theta}_j)$ bits
  across all $J$ tests.
- **Action**: Monitored on the central SRE dashboard. If $H_{\text{fleet}}$
  rises by $>15\%$ week-over-week, new non-critical feature merges are paused
  until the top entropy contributors are quarantined.

### Metric 2: CI Dollar Burn Efficiency ($\Delta \text{ USD}$)
- **Calculation**:

$$
\text{Monthly Savings} = (N_{\text{static}} - N_{\text{SPRT}}) \times \text{Cost per Run} \times \text{Daily Builds} \times 30
$$

  In a fleet with 20,000 daily builds and $\$0.20$ cost per run:

$$
(3.0 - 1.2) \times \$0.20 \times 20,000 \times 30 = \mathbf{\$216,000 \text{ / month in AWS EC2 compute savings}}
$$

### Metric 3: Regressions Escaped to Production (SLA Guardrail)
- **Calculation**: Escaped regressions per 1,000 merged PRs.
- **Action**: Guaranteed by setting the SPRT upper boundary $\beta_{\text{err}}
  \le 0.001$
  ($0.1\%$ escape limit).

### Metric 4: Quarantine Streak Exit Probability
- **Calculation**: $P(\text{pass next } 25 \text{ runs} \mid \alpha_t, \beta_t)
  \ge 0.90$.
- **Action**: Automated Slack notification to test owner: *"Test
  `test_order_stream` passed 30 soak runs. Bayesian 25-run predictive streak
  probability is 92.4%. Graduating back to blocking status."*
